In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

np.random.seed(42)

covtype = fetch_covtype()

X = covtype.data.astype(np.float32)
y = covtype.target.astype(np.int64)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of classes:", len(np.unique(y)))

class_counts = pd.Series(y).value_counts().sort_index()

print("\nClass Distribution:")
print(class_counts)

plt.figure(figsize=(8, 5))
plt.bar(class_counts.index, class_counts.values,color='purple')
plt.xlabel("Cover Type")
plt.ylabel("Number of Samples")
plt.title("Covertype Class Distribution")
plt.xticks(class_counts.index)
plt.show()

max_class_count = class_counts.max()
min_class_count = class_counts.min()

imbalance_ratio = max_class_count / min_class_count

print("\nLargest class:", max_class_count)
print("Smallest class:", min_class_count)
print("Imbalance ratio:", imbalance_ratio)

y = y - 1

num_classes = 7

Y = np.eye(num_classes, dtype=np.float32)[y]

print("\nFeature shape:", X.shape)
print("Target shape:", y.shape)
print("One-hot target shape:", Y.shape)

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("\nTraining:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

print("\nTraining mean:", np.mean(X_train))
print("Training std:", np.std(X_train))

N = len(y_train)
K = num_classes

class_counts_train = np.bincount(
    y_train,
    minlength=K
)

class_weights = N / (K * class_counts_train)

print("\nTraining Class Counts:")
print(class_counts_train)

print("\nClass Weights:")

for c in range(K):
    print(
        f"Class {c}: {class_weights[c]:.4f}"
    )


def relu(Z):
    return np.maximum(0, Z)


def relu_derivative(Z):
    return (Z > 0).astype(np.float32)


def softmax(Z):
    Z_shifted = Z - np.max(
        Z,
        axis=1,
        keepdims=True
    )

    exp_Z = np.exp(Z_shifted)

    return exp_Z / np.sum(
        exp_Z,
        axis=1,
        keepdims=True
    )


def he_initialization(
    input_size,
    output_size
):
    return (
        np.random.randn(
            input_size,
            output_size
        ).astype(np.float32)
        * np.sqrt(2.0 / input_size)
    )


W1 = he_initialization(54, 64)
b1 = np.zeros(
    (1, 64),
    dtype=np.float32
)

W2 = he_initialization(64, 32)
b2 = np.zeros(
    (1, 32),
    dtype=np.float32
)

W3 = he_initialization(32, 16)
b3 = np.zeros(
    (1, 16),
    dtype=np.float32
)

W4 = he_initialization(16, 7)
b4 = np.zeros(
    (1, 7),
    dtype=np.float32
)


def forward(
    X,
    W1,
    b1,
    W2,
    b2,
    W3,
    b3,
    W4,
    b4
):

    Z1 = X @ W1 + b1
    A1 = relu(Z1)

    Z2 = A1 @ W2 + b2
    A2 = relu(Z2)

    Z3 = A2 @ W3 + b3
    A3 = relu(Z3)

    Z4 = A3 @ W4 + b4
    A4 = softmax(Z4)

    cache = {
        "X": X,
        "Z1": Z1,
        "A1": A1,
        "Z2": Z2,
        "A2": A2,
        "Z3": Z3,
        "A3": A3,
        "Z4": Z4,
        "A4": A4
    }

    return A4, cache


def weighted_cross_entropy(
    Y_true,
    Y_pred,
    class_weights
):

    labels = np.argmax(
        Y_true,
        axis=1
    )

    sample_weights = class_weights[
        labels
    ]

    Y_pred = np.clip(
        Y_pred,
        1e-12,
        1.0
    )

    loss = -np.sum(
        sample_weights
        * np.sum(
            Y_true * np.log(Y_pred),
            axis=1
        )
    ) / np.sum(sample_weights)

    return loss


def backward(
    Y,
    cache,
    W2,
    W3,
    W4,
    class_weights
):

    X = cache["X"]

    Z1 = cache["Z1"]
    A1 = cache["A1"]

    Z2 = cache["Z2"]
    A2 = cache["A2"]

    Z3 = cache["Z3"]
    A3 = cache["A3"]

    A4 = cache["A4"]

    labels = np.argmax(
        Y,
        axis=1
    )

    sample_weights = class_weights[
        labels
    ].reshape(-1, 1)

    dZ4 = (
        (A4 - Y)
        * sample_weights
    )

    dZ4 /= np.sum(
        sample_weights
    )

    dW4 = A3.T @ dZ4
    db4 = np.sum(
        dZ4,
        axis=0,
        keepdims=True
    )

    dA3 = dZ4 @ W4.T
    dZ3 = (
        dA3
        * relu_derivative(Z3)
    )

    dW3 = A2.T @ dZ3
    db3 = np.sum(
        dZ3,
        axis=0,
        keepdims=True
    )

    dA2 = dZ3 @ W3.T
    dZ2 = (
        dA2
        * relu_derivative(Z2)
    )

    dW2 = A1.T @ dZ2
    db2 = np.sum(
        dZ2,
        axis=0,
        keepdims=True
    )

    dA1 = dZ2 @ W2.T
    dZ1 = (
        dA1
        * relu_derivative(Z1)
    )

    dW1 = X.T @ dZ1
    db1 = np.sum(
        dZ1,
        axis=0,
        keepdims=True
    )

    return (
        dW1,
        db1,
        dW2,
        db2,
        dW3,
        db3,
        dW4,
        db4
    )


learning_rate = 0.01
batch_size = 128
epochs = 100
patience = 15

history = {
    "train_loss": [],
    "val_loss": [],
    "train_accuracy": [],
    "val_accuracy": []
}

best_val_loss = np.inf
patience_counter = 0

best_W1 = W1.copy()
best_b1 = b1.copy()

best_W2 = W2.copy()
best_b2 = b2.copy()

best_W3 = W3.copy()
best_b3 = b3.copy()

best_W4 = W4.copy()
best_b4 = b4.copy()


for epoch in range(epochs):

    indices = np.random.permutation(
        len(X_train)
    )

    X_train_shuffled = X_train[
        indices
    ]

    y_train_shuffled = y_train[
        indices
    ]

    epoch_loss = 0.0
    num_batches = 0

    for start in range(
        0,
        len(X_train),
        batch_size
    ):

        end = start + batch_size

        X_batch = X_train_shuffled[
            start:end
        ]

        y_batch = y_train_shuffled[
            start:end
        ]

        Y_batch = np.eye(
            K,
            dtype=np.float32
        )[y_batch]

        predictions, cache = forward(
            X_batch,
            W1,
            b1,
            W2,
            b2,
            W3,
            b3,
            W4,
            b4
        )

        loss = weighted_cross_entropy(
            Y_batch,
            predictions,
            class_weights
        )

        epoch_loss += loss
        num_batches += 1

        gradients = backward(
            Y_batch,
            cache,
            W2,
            W3,
            W4,
            class_weights
        )

        (
            dW1,
            db1,
            dW2,
            db2,
            dW3,
            db3,
            dW4,
            db4
        ) = gradients

        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1

        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2

        W3 -= learning_rate * dW3
        b3 -= learning_rate * db3

        W4 -= learning_rate * dW4
        b4 -= learning_rate * db4

    train_loss = (
        epoch_loss / num_batches
    )

    val_probabilities, _ = forward(
        X_val,
        W1,
        b1,
        W2,
        b2,
        W3,
        b3,
        W4,
        b4
    )

    Y_val = np.eye(
        K,
        dtype=np.float32
    )[y_val]

    val_loss = weighted_cross_entropy(
        Y_val,
        val_probabilities,
        class_weights
    )

    train_probabilities, _ = forward(
        X_train,
        W1,
        b1,
        W2,
        b2,
        W3,
        b3,
        W4,
        b4
    )

    train_predictions = np.argmax(
        train_probabilities,
        axis=1
    )

    val_predictions = np.argmax(
        val_probabilities,
        axis=1
    )

    train_accuracy = accuracy_score(
        y_train,
        train_predictions
    )

    val_accuracy = accuracy_score(
        y_val,
        val_predictions
    )

    history["train_loss"].append(
        train_loss
    )

    history["val_loss"].append(
        val_loss
    )

    history["train_accuracy"].append(
        train_accuracy
    )

    history["val_accuracy"].append(
        val_accuracy
    )

    print(
        f"Epoch {epoch + 1:03d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        patience_counter = 0

        best_W1 = W1.copy()
        best_b1 = b1.copy()

        best_W2 = W2.copy()
        best_b2 = b2.copy()

        best_W3 = W3.copy()
        best_b3 = b3.copy()

        best_W4 = W4.copy()
        best_b4 = b4.copy()

    else:

        patience_counter += 1

        if patience_counter >= patience:

            print(
                "\nEarly stopping triggered."
            )

            break


W1 = best_W1
b1 = best_b1

W2 = best_W2
b2 = best_b2

W3 = best_W3
b3 = best_b3

W4 = best_W4
b4 = best_b4


plt.figure(figsize=(8, 5))

plt.plot(
    history["train_loss"],
    label="Training Loss"
)

plt.plot(
    history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(
    "Training and Validation Loss"
)

plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(8, 5))

plt.plot(
    history["train_accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title(
    "Training and Validation Accuracy"
)

plt.legend()
plt.grid(True)
plt.show()


test_probabilities, _ = forward(
    X_test,
    W1,
    b1,
    W2,
    b2,
    W3,
    b3,
    W4,
    b4
)

y_pred = np.argmax(
    test_probabilities,
    axis=1
)

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    average=None,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average=None,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average=None,
    zero_division=0
)

macro_precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

print("\n" + "=" * 60)
print("FINAL MODEL PERFORMANCE")
print("=" * 60)

print(
    f"Accuracy        : {accuracy:.4f}"
)

print(
    f"Macro Precision : {macro_precision:.4f}"
)

print(
    f"Macro Recall    : {macro_recall:.4f}"
)

print(
    f"Macro F1        : {macro_f1:.4f}"
)

results = pd.DataFrame({
    "Class": np.arange(1, 8),
    "Precision": precision,
    "Recall": recall,
    "F1-Score": f1
})

print("\nPer-Class Results")
print(results)

print("\nClassification Report")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Class 1",
            "Class 2",
            "Class 3",
            "Class 4",
            "Class 5",
            "Class 6",
            "Class 7"
        ],
        zero_division=0
    )
)


cm = confusion_matrix(
    y_test,
    y_pred
)

print("\nConfusion Matrix")
print(cm)


plt.figure(figsize=(8, 7))

plt.imshow(cm)

plt.colorbar()

plt.xticks(
    np.arange(7),
    np.arange(1, 8)
)

plt.yticks(
    np.arange(7),
    np.arange(1, 8)
)

plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.title(
    "7 × 7 Confusion Matrix"
)

for i in range(7):

    for j in range(7):

        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.tight_layout()
plt.show()


plt.figure(figsize=(8, 5))

plt.bar(
    np.arange(1, 8),
    f1
)

plt.xlabel("Class")
plt.ylabel("F1-Score")
plt.title(
    "Per-Class F1 Scores"
)

plt.xticks(
    np.arange(1, 8)
)

plt.ylim(
    0,
    1
)

plt.grid(
    axis="y"
)

plt.show()


plt.figure(figsize=(8, 5))

plt.scatter(
    recall,
    precision,
    s=100
)

for i in range(7):

    plt.annotate(
        f"Class {i + 1}",
        (recall[i], precision[i]),
        xytext=(5, 5),
        textcoords="offset points"
    )

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(
    "Precision vs Recall by Class"
)

plt.xlim(0, 1)
plt.ylim(0, 1)

plt.grid(True)

plt.show()


true_counts = np.bincount(
    y_test,
    minlength=7
)

pred_counts = np.bincount(
    y_pred,
    minlength=7
)

distribution = pd.DataFrame({
    "Class": np.arange(1, 8),
    "True": true_counts,
    "Predicted": pred_counts
})

print("\nTrue vs Predicted Distribution")
print(distribution)


x = np.arange(7)
width = 0.35

plt.figure(figsize=(9, 5))

plt.bar(
    x - width / 2,
    true_counts,
    width,
    label="True"
)

plt.bar(
    x + width / 2,
    pred_counts,
    width,
    label="Predicted"
)

plt.xlabel("Class")
plt.ylabel("Number of Samples")
plt.title(
    "True vs Predicted Class Distribution"
)

plt.xticks(
    x,
    np.arange(1, 8)
)

plt.legend()
plt.grid(axis="y")

plt.show()